# Trailwise · From a trek record to an answer
In Google Colab, choose **Runtime → Run all** with the default CPU runtime. The first cell downloads the public coursework files and installs the libraries for the walkthrough. No API key, model download or Google Drive mount is needed.

You will validate the public data, run a search and inspect the prompt built from its results. The side-by-side model answers later in the notebook were recorded during evaluation. The optional Mac service call stays off during a normal Colab run.

In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

# Use the local project when available. Colab opens the notebook by itself.
manifest_path = Path("documents/abc-six-day/manifest.json")
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / manifest_path).is_file()), None)
if root is None:
    if "download_folder" not in globals():
        download_folder = TemporaryDirectory(prefix="trailwise-")
    root = Path(download_folder.name) / "trailwise-abc"
    if not (root / ".git").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--no-checkout",
                        "https://github.com/Chen-Xingjian/trailwise-abc.git", str(root)], check=True)
    # Keep the app, dataset and archived answers on the same published revision.
    subprocess.run(["git", "-C", str(root), "checkout", "--quiet", "--detach",
                    "27dbc546ea8aa7c05222d43cfbf108ef9dd3ec49"], check=True)

# Install only the packages used for CPU retrieval and prompt construction.
packages = {"qdrant_client": "qdrant-client>=1.13,<2", "numpy": "numpy>=1.26,<3",
            "httpx": "httpx>=0.27,<1", "dotenv": "python-dotenv>=1,<2"}
if any(importlib.util.find_spec(module) is None for module in packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages.values()],
                   check=True)
    importlib.invalidate_caches()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Ready to load:", root / manifest_path)

## 1 · Validate the documents
The same manifest governs the ABC web launcher and this notebook. The loader checks file hashes, source rights reviews, accepted IDs and dependencies before constructing chunks.

In [ ]:
from app.config import Settings
from app.release import load_release
from app.service import Assistant

corpus = root / 'documents/abc-six-day'
release = load_release(corpus)
print(release.manifest['release_id'])
print(f'{len(release.facts)} accepted facts, {len(release.sources)} source records, {len(release.chunks)} chunks')

## 2 · Inspect a chunk and its attribution
Each chunk preserves one statement, its itinerary reference and its qualification. The attached source locator lets a reviewer trace it to the original publication. Here the subject is moving time on the return from ABC to Bamboo.

In [ ]:
by_id = {chunk['fact_id']: chunk for chunk in release.chunks}
example = by_id['KT-D5-M']
print(example['text'])
for source in example['source_records']:
    print(source['source_title'], source['source_locator'], source['licence_id'], sep=' | ')

## 3 · Retrieve evidence and inspect the prompt
This exercise calls the application’s retriever and prompt builder. A lightweight hash embedding keeps the example runnable offline without model weights. The frozen semantic run used Qwen embeddings. Compare its actual top three in the next section.

In [ ]:
rows = json.loads((root / 'outputs/r38/pair-results.json').read_text())
def pair(case, turn):
    return {row['mode']: row for row in rows if row['id'] == case and row['turn'] == turn}

recorded = pair('Z01', 1)
question = recorded['rag']['question']
with TemporaryDirectory() as temp:
    preview = Assistant(Settings(release_dir=corpus, runtime_dir=Path(temp), embedding_mode='hash'))
    try:
        hits = preview.retriever.search(question, limit=3)
        messages = preview.messages(question, hits)
    finally:
        preview.retriever.close()

print('Preview retrieval:', [hit['fact_id'] for hit in hits])
print(messages[0]['content'])
print(messages[1]['content'])

## 4 · Compare recorded first responses
These are archived answers from R38 evaluating candidate R37. Z01 asks about moving versus elapsed time. Z03 turn two tests whether a confirmed five-day walking budget resolves the opening question. Examine whether the answer uses the retrieved evidence and the user’s latest information.

In [ ]:
for case, turn in [('Z01', 1), ('Z03', 2)]:
    arms = pair(case, turn)
    print(case, 'turn', turn, arms['rag']['question'], sep=' | ')
    for name in ('rag', 'baseline'):
        row = arms[name]
        print(name, 'retrieved', row['retrieved'], 'score', row['score'], '/ 2')
        print(row['answer'])
    print()

## 5 · Optional live comparison on Mac
Leave `RUN_LIVE = False` on a Google-hosted runtime. The walkthrough is complete after the recorded comparison. To generate a new pair, run the ABC launcher on a Mac, use a notebook kernel on that same Mac and set the switch to True. The app and notebook must load the same release. Live weather and notice checks are disabled for this comparison.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    from urllib.request import ProxyHandler, Request, build_opener

    local = build_opener(ProxyHandler({}))
    base = 'http://127.0.0.1:8765'
    with local.open(base + '/api/status', timeout=30) as response:
        status = json.load(response)
    if (status['release_id'] != release.manifest['release_id']
            or status['indexed_fact_count'] != len(release.facts)
            or status['embedding']['mode'] != 'sentence-transformers'
            or not status['model']['connected']):
        raise RuntimeError('Start the ABC launcher with the reviewed corpus and semantic model.')
    for mode in ('rag', 'baseline'):
        payload = {'question': question, 'mode': mode, 'top_k': 3, 'offer_live_checks': False}
        request = Request(base + '/api/ask', data=json.dumps(payload).encode(),
                          headers={'Content-Type': 'application/json'})
        with local.open(request, timeout=180) as response:
            result = json.load(response)
        print(mode, result['answer_kind'], [hit['fact_id'] for hit in result['evidence']])
        print(result['answer'])
else:
    print('Live generation is off. The two comparisons above are archived outputs.')